In [1]:
from langchain_openai import OpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

In [2]:
ALLOW_DANGEROUS_REQUEST = True


In [ ]:
from typing import Any, Dict, Union

import requests
import yaml


def _get_schema(response_json: Union[dict, list]) -> dict:
    if isinstance(response_json, list):
        response_json = response_json[0] if response_json else {}
    return {key: type(value).__name__ for key, value in response_json.items()}


def _get_api_spec() -> str:
    base_url = "https://jsonplaceholder.typicode.com"
    endpoints = [
        "/posts",
        "/comments",
    ]
    common_query_parameters = [
        {
            "name": "_limit",
            "in": "query",
            "required": False,
            "schema": {"type": "integer", "example": 2},
            "description": "Limit the number of results",
        }
    ]
    openapi_spec: Dict[str, Any] = {
        "openapi": "3.0.0",
        "info": {"title": "JSONPlaceholder API", "version": "1.0.0"},
        "servers": [{"url": base_url}],
        "paths": {},
    }
    # Iterate over the endpoints to construct the paths
    for endpoint in endpoints:
        response = requests.get(base_url + endpoint)
        if response.status_code == 200:
            schema = _get_schema(response.json())
            openapi_spec["paths"][endpoint] = {
                "get": {
                    "summary": f"Get {endpoint[1:]}",
                    "parameters": common_query_parameters,
                    "responses": {
                        "200": {
                            "description": "Successful response",
                            "content": {
                                "application/json": {
                                    "schema": {"type": "object", "properties": schema}
                                }
                            },
                        }
                    },
                }
            }
    return yaml.dump(openapi_spec, sort_keys=False)


api_spec = _get_api_spec()

In [3]:
from langchain_community.agent_toolkits.openapi.toolkit import RequestsToolkit
from langchain_community.utilities.requests import TextRequestsWrapper
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://google.com"
    }

toolkit = RequestsToolkit(
    requests_wrapper=TextRequestsWrapper(headers=headers),
    allow_dangerous_requests=ALLOW_DANGEROUS_REQUEST,
)

In [5]:
tools = toolkit.get_tools()

tools

[RequestsGetTool(requests_wrapper=TextRequestsWrapper(headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36', 'Accept-Language': 'en-US,en;q=0.9', 'Referer': 'https://google.com'}, aiosession=None, auth=None, response_content_type='text', verify=True), allow_dangerous_requests=True),
 RequestsPostTool(requests_wrapper=TextRequestsWrapper(headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36', 'Accept-Language': 'en-US,en;q=0.9', 'Referer': 'https://google.com'}, aiosession=None, auth=None, response_content_type='text', verify=True), allow_dangerous_requests=True),
 RequestsPatchTool(requests_wrapper=TextRequestsWrapper(headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36', 'Accept-Language': 'en-US,en;q=0.9', 'Referer': 'https://google.com'}, 

In [5]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
base_url = "http://127.0.0.1:1234/v1" 
model_name ="google/gemma-3-4b"
#model_name ="qwen/qwen3-1.7b"
llm = ChatOpenAI(model=model_name, temperature=0, max_tokens=None, timeout=None, max_retries=2,api_key="llm-studio", base_url=base_url)


In [ ]:


system_message = """
You have access to an API to help answer user queries.
Here is documentation on the API:
{api_spec}
""".format(api_spec=api_spec)

agent_executor = create_react_agent(llm, tools, prompt=system_message)

In [ ]:
example_query = "Fetch the top two posts. What are their titles?"

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

In [6]:
from bs4 import BeautifulSoup
import re

def extract_date_candidates(html: str) -> list[str]:
    """
    Extract candidate date strings from HTML using BeautifulSoup + regex.
    Returns a list of strings like '2025-09-19', '19 Sep 2025', etc.
    """
    soup = BeautifulSoup(html, "html.parser")
    candidates = set()

    for tag in soup.find_all(["meta", "time", "span", "div"]):
        # Meta tags with date attributes
        if tag.name == "meta":
            for attr in ["property", "name", "itemprop"]:
                if tag.has_attr(attr) and "date" in tag[attr].lower():
                    if tag.has_attr("content"):
                        candidates.add(tag["content"])

        # Time tags
        if tag.name == "time":
            if tag.has_attr("datetime"):
                candidates.add(tag["datetime"])
            candidates.add(tag.get_text(strip=True))

        # Visible text regex matches
        text = tag.get_text(" ", strip=True)
        if re.search(r"\b\d{1,2}\s*(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s*\d{4}\b", text, re.I):
            candidates.add(text)
        if re.search(r"\b\d{4}-\d{2}-\d{2}\b", text):  # ISO
            candidates.add(text)

    return list(candidates)

tools_custom = [
    {
        "name": "extract_date_candidates",
        "description": "Extract possible publish date strings from raw HTML content.",
        "parameters": {
            "html": {
                "type": "string",
                "description": "The full HTML source of the article"
            }
        },
        "function": extract_date_candidates,
    }
]

#tools = tools + tools_custom

system_prompt = """
You are a date extraction assistant.

Goal:
- Extract the publish date of an online news article from its HTML given url.

Steps:
1. Use Request Toolkit to fetch html content from url
1. Use the `extract_date_candidates` tool to pull possible date strings from hmtl content.
2. Choose the most likely publish date (ignore unrelated dates like 'last updated').
3. Return only the date in strict ISO format: YYYY-MM-DD.
4. If no valid date exists, return "null".
"""

from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI  # or your preferred LLM

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
base_url = "http://127.0.0.1:1234/v1" 
model_name ="google/gemma-3-4b"
#model_name ="qwen/qwen3-1.7b"
llm = ChatOpenAI(model=model_name, temperature=0, max_tokens=None, timeout=None, max_retries=2,api_key="llm-studio", base_url=base_url)

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)


# Example run
url = "https://www.newsweek.com/elon-musk-x-latest-posts-tweets-update-2047984"
#result = agent.invoke({"input": f"Extract the publish date from this url:\n{url}"})
#print("Final Date:", result["output"])
query = f"Extract the publish date from this url:\n{url}"
result = agent.invoke({"messages": [("user", query)]} )
print(result["messages"][-1].content)


GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/GRAPH_RECURSION_LIMIT

In [7]:
agent_executor = create_react_agent(llm, tools)
example_query = "what is content of post https://www.newsweek.com/elon-musk-x-latest-posts-tweets-update-2047984"
example_query = "Navigate to https://en.wikipedia.org/wiki/English_Wikipedia and tell me the main heading."
query = '''You are given extracted HTML content from a news article. 
Find and return the publish date and updated date in ISO 8601 format (YYYY-MM-DD).
If no date is found, return "null".

url : https://www.newsweek.com/elon-musk-x-latest-posts-tweets-update-2047984
'''

events = agent_executor.stream(
     {"messages": [("user", query)]},
    stream_mode="updates"
)
for event in events:
    print(event)
    #event["messages"][-1].pretty_print()

{'agent': {'messages': [AIMessage(content='I can’t access external websites or specific URLs to extract information from them. Therefore, I cannot fulfill your request to find and return the publish date and updated date in ISO 8601 format from the provided URL.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 1270, 'total_tokens': 1317, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-fy7numvjm8ksql9woro60p', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--6b608707-c7ad-4bbd-83a1-067c30952f28-0', usage_metadata={'input_tokens': 1270, 'output_tokens': 47, 'total_tokens': 1317, 'input_token_details': {}, 'output_token_details': {}})]}}


In [6]:
agent_executor = create_react_agent(llm, tools)
query = """Anaylse the below url  to find out when the article is published. 
       Use the infomation in post alone to find published date, don't assume things yourself.  
       
       url : https://www.newsweek.com/elon-musk-x-latest-posts-tweets-update-2047984
"""
query = '''You are given extracted HTML content from a news article. 
Find and return the publish date and updated date in ISO 8601 format (YYYY-MM-DD).
If no date is found, return "null".

url : https://www.newsweek.com/elon-musk-x-latest-posts-tweets-update-2047984
'''

#query = "https://www.newsweek.com/elon-musk-x-latest-posts-tweets-update-2047984"
#query = "Navigate to https://en.wikipedia.org/wiki/English_Wikipedia and tell me the main heading."
#query = "What are the headers on langchain.com?"
result = agent_executor.invoke({"messages": [("user", query)]} )
print(result["messages"][-1].content)


I can’t access external websites or specific URLs to extract information from them. Therefore, I cannot fulfill your request to find and return the publish date and updated date in ISO 8601 format from the provided URL.
